In [101]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [102]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [103]:
from rag_helper import RAGBase
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [104]:
answer=assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also run:
```bash
curl http://localhost:11434
```

If needed, you can install the Python client with:
```bash
pip install ollama
```


In [105]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [106]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

In [107]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [108]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment registration course FAQ"}', call_id='call_Xvv0zW5QrhvlP362fufjRpJW', name='search', type='function_call', id='fc_06e47b36f4d46568006a15c906be4c8197853f7231219b0dc5', namespace=None, status='completed')]

In [109]:
call = response.output[0]

In [110]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment registration course FAQ"}', call_id='call_Xvv0zW5QrhvlP362fufjRpJW', name='search', type='function_call', id='fc_06e47b36f4d46568006a15c906be4c8197853f7231219b0dc5', namespace=None, status='completed')

In [111]:
import json
args = json.loads(call.arguments)
args

{'query': 'join course discovered can I join enrollment registration course FAQ'}

In [112]:
result = search(**args)
result_json = json.dumps(result, indent=2)

In [113]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json

}

In [114]:
messages.append(call)
messages.append(function_call_output)

In [115]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment registration course FAQ"}', call_id='call_Xvv0zW5QrhvlP362fufjRpJW', name='search', type='function_call', id='fc_06e47b36f4d46568006a15c906be4c8197853f7231219b0dc5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Xvv0zW5QrhvlP362fufjRpJW',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a cer

In [116]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool],
)


In [117]:
response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [118]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [119]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()
question = "I just discovered the course. Can I join it?"
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}

]

In [120]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool],
)


In [121]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course enroll late registration discovered the course can I join it"}', call_id='call_NUQUzM81tnrV8Kl9ak920u47', name='search', type='function_call', id='fc_064d8878c8f97a26006a15c9095fbc81939498e6380f3de340', namespace=None, status='completed')]

In [122]:
messages.extend(response.output)

for item in response.output:
    if item.type    == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == 'message':
        print("Assistant message:")
        print( item.content[0].text)

function_call: search {"query":"join course enroll late registration discovered the course can I join it"}


In [123]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches if needed. Try to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll late registration discovered the course can I join it"}', call_id='call_NUQUzM81tnrV8Kl9ak920u47', name='search', type='function_call', id='fc_064d8878c8f97a26006a15c9095fbc81939498e6380f3de340', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_NUQUzM81tnrV8Kl9ak920u47',
  'output': '[\n  {\n    "id": 

In [132]:
def agent_loop(instructions, question, model='gpt-5.4-mini'):
        messages = [
            {'role': 'developer', 'content': instructions},
            {'role': 'user', 'content': question}

        ]

        it=1

        while True:
            print(f'iteration #{it}.....')
            response = openai_client.responses.create(
                model='gpt-5.4-mini',
                input=messages,
                tools=[search_tool],
            )

            messages.extend(response.output)

            has_function_calls = False

            for entry in response.output:
                if entry.type == 'message':
                    print("Assistant message:")
                    last_answer= entry.content[0].text
                    print(entry.content[0].text)

                if entry.type == 'function_call':
                    print('function_call:', entry.name, entry.arguments)
                    result = make_call(entry)
                    messages.append(result)
                    has_function_calls = True


            it = it+1

            if not has_function_calls:
                break
        return last_answer

In [133]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. First perform serach then analyze the results and perform more searches based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()
question = "I just discovered the course. Can I join it?"

result = agent_loop(instructions, question)

iteration #1.....
function_call: search {"query":"join course enrollment discovered the course can I join"}
iteration #2.....
Assistant message:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you only want to learn, you can start anytime.

If you’d like, I can also help with how certificates, homework, or the course schedule work.


In [134]:
response.output

[ResponseOutputMessage(id='msg_05e72d8a0f0e0a3f006a15cb60ddc881958747192141a929fa', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.  \n\nIf you want a certificate, though, you need to submit your project while submissions are still open.\n\nIf you’d like, I can also explain whether registration is required or how the certificate process works. Any other areas you want to explore?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]